In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split, TensorDataset, Subset
from torchvision import datasets, transforms
from functools import partial
import os
import random
import numpy as np

SEED = 0
torch.manual_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
g = torch.Generator()
g.manual_seed(0)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'  # or ':16:8'
    os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
    torch.set_deterministic_debug_mode(True)

################################
#     RESTART     RUNTIME      #
################################
from ssonn.metrics.nonlinearity_metrics import *
from ssonn.metrics.edge_finder import *
from ssonn.train.train import *

import torchvision.models as models


In [2]:

import warnings
warnings.filterwarnings("ignore", category=UserWarning)


In [4]:

BATCH_SIZE = 256

# CIFAR-100 normalization
CIFAR100_MEAN = (0.5071, 0.4867, 0.4408)
CIFAR100_STD  = (0.2675, 0.2565, 0.2761)

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])

full_train_dataset = datasets.CIFAR100(
    root='./data', train=True, download=True, transform=transform_train
)

test_dataset = datasets.CIFAR100(
    root='./data', train=False, download=True, transform=transform_test
)

# simplest baseline: use official test set as validation
train_dataset = full_train_dataset
val_dataset = test_dataset

# If you prefer a real split:
# train_dataset, val_dataset = random_split(full_train_dataset, [0.9, 0.1], generator=g)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# loader used by edge-finder / metric aggregation (no shuffle)
finder_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)


100%|██████████| 169M/169M [00:40<00:00, 4.18MB/s] 


In [5]:

class ExpandingHead(nn.Module):
    """
    Backbone is a CNN feature extractor.
    We apply SSONN only to the classification head (fc0).
    """
    def __init__(self, num_classes=100):
        super(ExpandingHead, self).__init__()
        backbone = models.vgg19_bn(weights=None)
        self.backbone = backbone.features
        self.fc0 = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.backbone(x)
        x = torch.flatten(x, 1)
        x = self.fc0(x)
        return x

    def freeze_backbone(self):
        for param in self.backbone.parameters():
            param.requires_grad = False


In [6]:

def init_weights(m):
    if isinstance(m, nn.Conv2d):
        nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)

    elif isinstance(m, nn.BatchNorm2d):
        nn.init.constant_(m.weight, 1)
        nn.init.constant_(m.bias, 0)

    elif isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)


In [7]:

model = ExpandingHead(num_classes=100)
model.apply(init_weights)

# Convert only the head to a scaled/expandable layer
model = convert_to_scaled_net(model, layers_to_convert=[model.fc0], device=device)

# Optional: freeze backbone
# model.freeze_backbone()


Converting layer 'fc0' to ScaledNet...


In [8]:

hyperparams = {
    "num_epochs": 448,
    "batch_size": BATCH_SIZE,
    "edge_importance_metric": AbsGradientEdgeMetric(nn.CrossEntropyLoss()),
    "edge_score_aggregation": "variance",
    "expansion_thresholds": {"fc0": 0.002},
    "pruning_thresholds": {"fc0": 1},
    "plateau_threshold": 0.01,
    "min_epochs_between_expansions": 12,
    "plateau_window_size": 5,
    "learning_rate": 1e-2,
    "weight_decay": 0,
    "prune_after_epochs": 4,
    "task_type": "classification",
    "start_fully_connected": False,
    "max_new_edges_per_expansion": 10_000,
    "diff_threshold": 0.05,
    "t_max": 50,
    "warmup_epochs": 5
}


In [9]:

from comet_ml import start
from comet_ml.integration.pytorch import log_model

experiment = start(
    api_key=os.environ.get("COMET_API_KEY"),
    workspace=os.environ.get("COMET_WORKSPACE"),
    project_name="cifar100-backbone",
)
experiment.log_parameters(hyperparams)
experiment.set_name("cifar100_vgg19bn_expand_head")


COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: sklearn, torch.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/gcof/cifar100-backbone/210527ed9ba045e4bb39de03fb435ae9



In [ ]:

criterion = nn.CrossEntropyLoss()

def optimizer_factory(model_instance, current_lr=None, base_lr=None):
    if base_lr is None:
        base_lr = hyperparams['learning_rate']
    if current_lr is None:
        current_lr = base_lr

    new_optimizer = optim.Adam(
        model_instance.parameters(),
        lr=current_lr,
        weight_decay=hyperparams['weight_decay'],
    )

    for group in new_optimizer.param_groups:
        group['initial_lr'] = base_lr

    return new_optimizer


trained_model = train_ssonn(
    model,
    train_loader,
    val_loader,
    finder_loader,
    criterion,
    optimizer_factory,
    hyperparams,
    device,
    experiment,
    True
)


--- Starting SSONN Training ---


Epoch 1/448 | T-Loss: 5.4308 | V-Loss: 5.1237 | Acc: 0.0115 | LRs: [1.0e-02] | Params: 51300/51300


Epoch 2/448 | T-Loss: 4.6896 | V-Loss: 5.2810 | Acc: 0.0197 | LRs: [1.0e-02] | Params: 51300/51300


Epoch 3/448 | T-Loss: 4.3985 | V-Loss: 4.3283 | Acc: 0.0365 | LRs: [9.9e-03] | Params: 51300/51300


Epoch 4/448 | T-Loss: 4.2863 | V-Loss: 4.3140 | Acc: 0.0343 | LRs: [9.8e-03] | Params: 51300/51300


Epoch 5/448 | T-Loss: 4.1775 | V-Loss: 4.1887 | Acc: 0.0525 | LRs: [9.8e-03] | Params: 51300/51300


... (8 outputs left out)

Epoch 14/448 | T-Loss: 3.0018 | V-Loss: 3.1455 | Acc: 0.1960 | LRs: [8.2e-03] | Params: 51300/51300


Epoch 15/448 | T-Loss: 2.8263 | V-Loss: 2.8650 | Acc: 0.2465 | LRs: [7.9e-03] | Params: 51300/51300


Epoch 16/448 | T-Loss: 2.6771 | V-Loss: 2.7783 | Acc: 0.2758 | LRs: [7.7e-03] | Params: 51300/51300


Epoch 17/448 | T-Loss: 2.5567 | V-Loss: 2.7692 | Acc: 0.2735 | LRs: [7.4e-03] | Params: 51300/51300


In [ ]:
experiment.end()